# Bank Distress Early-Warning Model: Feature Review & Cleaning
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

Picks up from the modeling table built in `feature_selection.ipynb`. Two jobs here:

1. **Understand every feature**, what each column means, where it comes from, how complete it is, and whether the literature backs it (`../literature/features_by_dataset.md`).
2. **Clean**, work the open items in `data_concerns.md` and run range/outlier checks.

Output: `../data/processed/panel_clean.parquet`, the table modeling reads.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("..") / "data" / "raw"
PROCESSED = Path("..") / "data" / "processed"

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

## Load the panel

The modeling table from `feature_selection.ipynb`: financials + FRED macro + static bank attributes, 1990Q1 forward, with the PCA tier and the `onset_4q` target already built.

In [2]:
panel = pd.read_parquet(PROCESSED / "panel.parquet")

print(f"panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols "
      f"({panel['REPDTE'].min():%Y-%m} to {panel['REPDTE'].max():%Y-%m})")
print(f"banks: {panel['CERT'].nunique():,}")

panel: 1,258,888 rows x 64 cols (1990-03 to 2026-03)
banks: 19,474


## Load the supporting tables

Needed for the cleaning work, not in the panel itself:

- `history`, charter events, to separate banks that merged away from banks that stayed healthy (concern #3)
- `failures`, actual FDIC failures, to sanity-check the distress label (concern #2)
- `institutions`, full static attributes, for the feature dictionary

In [3]:
history = pd.read_parquet(RAW / "history.parquet")
failures = pd.read_parquet(RAW / "failures.parquet")
failures["FAILDATE"] = pd.to_datetime(failures["FAILDATE"])
institutions = pd.read_parquet(RAW / "institutions.parquet")

for name, df in [("history", history), ("failures", failures),
                 ("institutions", institutions)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

history: 583,066 rows x 176 cols
failures: 4,115 rows x 22 cols
institutions: 27,836 rows x 149 cols


## Feature dictionary

One row per column in the panel: plain-English meaning, CAMELS category, source table, and whether the literature backs it as a predictor (`../literature/features_by_dataset.md`). Missing %, range, and other stats get computed and merged in the next cell, so this stays a clean reference map.

`role` marks how each column is used: **id** (key/name), **feature** (model input), **label** (target or its building blocks), **derived** (built in prep). Lit backing applies to features only.

In [4]:
# name: (plain english, category, source, role, lit-backed feature?)
# lit backing per ../literature/features_by_dataset.md and lit_notes.md (Cole & White
# is the headline feature-selection source). 'lit' only meaningful for role == 'feature'.
FEATURE_DICT = {
    # --- identifiers ---
    "CERT":     ("FDIC certificate number (bank id)", "ID", "financials", "id", ""),
    "REPDTE":   ("Reporting quarter-end date", "ID", "financials", "id", ""),
    "NAME":     ("Bank name", "ID", "financials", "id", ""),
    "ASSET":    ("Total assets ($000)", "Size", "financials", "feature", "yes"),
    "DEP":      ("Total deposits ($000)", "Size", "financials", "feature", "yes"),

    # --- capital (also the PCA label source) ---
    "RBC1AAJ":  ("Tier 1 leverage ratio (% avg assets)", "Capital", "financials", "feature", "yes"),
    "RBC1RWAJ": ("Tier 1 risk-based capital ratio (%)", "Capital", "financials", "feature", "yes"),
    "RBCRWAJ":  ("Total risk-based capital ratio (%)", "Capital", "financials", "feature", "yes"),
    "RBCT1CER": ("Common equity tier 1 ratio (%)", "Capital", "financials", "feature", "yes"),
    "RBCT1J":   ("Tier 1 (core) capital ($000)", "Capital", "financials", "feature", "yes"),
    "EQV":      ("Equity capital / assets (%)", "Capital", "financials", "feature", "yes"),
    "EQ":       ("Total equity capital ($000)", "Capital", "financials", "feature", "yes"),
    "econ_insolvent": ("Book equity <= 2% of assets (incl. unrealized losses)", "Capital", "derived", "feature", "yes"),

    # --- asset quality ---
    "NPERFV":   ("Nonperforming assets / assets (%)", "Asset quality", "financials", "feature", "yes"),
    "NCLNLSR":  ("Noncurrent loans / gross loans (%)", "Asset quality", "financials", "feature", "yes"),
    "NTLNLSR":  ("Net charge-offs / loans (%)", "Asset quality", "financials", "feature", "yes"),
    "LNATRESR": ("Loan loss reserves / loans (%)", "Asset quality", "financials", "feature", "yes"),
    "P3ASSETR": ("Past-due 30-89 days / assets (%)", "Asset quality", "financials", "feature", "yes"),
    "P9ASSETR": ("Past-due 90+ days / assets (%)", "Asset quality", "financials", "feature", "yes"),
    "ORER":     ("Other real estate owned / assets (%)", "Asset quality", "financials", "feature", "yes"),
    "ELNATRR":  ("Provision for loan losses / assets (%)", "Asset quality", "financials", "feature", "yes"),

    # --- loan portfolio mix (Cole & White headline) ---
    "LNRECONSR":("Construction & land dev loans / assets (%)", "Loan mix", "financials", "feature", "yes"),
    "LNRENRESR":("Nonfarm nonresidential (CRE) loans / assets (%)", "Loan mix", "financials", "feature", "yes"),
    "LNREMULTR":("Multifamily mortgage loans / assets (%)", "Loan mix", "financials", "feature", "yes"),
    "LNRERESR": ("1-4 family residential loans / assets (%)", "Loan mix", "financials", "feature", "yes"),
    "IDNCCIR":  ("Commercial & industrial loans / assets (%)", "Loan mix", "financials", "feature", "yes"),
    "LNCONR":   ("Consumer loans / assets (%)", "Loan mix", "financials", "feature", "yes"),

    # --- liquidity / funding ---
    "DEPUNA":   ("Uninsured deposits ($000)", "Funding", "financials", "feature", "yes"),
    "ESTINS":   ("Estimated insured deposits ($000)", "Funding", "financials", "feature", "yes"),
    "BRO":      ("Brokered deposits ($000)", "Funding", "financials", "feature", "yes"),
    "BROR":     ("Brokered deposits / total deposits (%)", "Funding", "financials", "feature", "yes"),
    "LNLSDEPR": ("Loans+leases / deposits (%)", "Funding", "financials", "feature", "yes"),
    "CHBALR":   ("Cash & balances due / assets (%)", "Funding", "financials", "feature", "yes"),
    "SCHA":     ("Held-to-maturity securities ($000)", "Funding", "financials", "feature", "partial"),
    "SCHF":     ("HTM securities, fair value ($000)", "Funding", "financials", "feature", "partial"),
    "SCAA":     ("Available-for-sale securities ($000)", "Funding", "financials", "feature", "partial"),
    "SCAF":     ("AFS securities, fair value ($000)", "Funding", "financials", "feature", "partial"),

    # --- earnings ---
    "ROA":      ("Return on assets (%)", "Earnings", "financials", "feature", "yes"),
    "ROE":      ("Return on equity (%)", "Earnings", "financials", "feature", "yes"),
    "NIMY":     ("Net interest margin (%)", "Earnings", "financials", "feature", "yes"),
    "EEFFR":    ("Efficiency ratio (%), also mgmt proxy", "Earnings", "financials", "feature", "yes")    "NETINC":   ("Net income ($000)", "Earnings", "financials", "feature", "yes"),

    # --- static institution attributes ---
    "ESTYMD":   ("Bank establishment date", "Static", "institutions", "derived", ""),
    "STALP":    ("State (2-letter)", "Static", "institutions", "feature", "yes"),
    "BKCLASS":  ("Charter/class (SM/NM/N/SB/...)", "Static", "institutions", "feature", "yes"),
    "REGAGNT":  ("Primary federal regulator", "Static", "institutions", "feature", "yes"),
    "AGE_YEARS":("Bank age in years at quarter", "Static", "derived", "feature", "yes"),

    # --- FRED macro context ---
    "FEDFUNDS": ("Fed funds rate (%)", "Macro", "FRED", "feature", "yes"),
    "DGS10":    ("10-year Treasury yield (%)", "Macro", "FRED", "feature", "yes"),
    "T10Y3M":   ("10yr-3mo term spread (%)", "Macro", "FRED", "feature", "yes"),
    "UNRATE":   ("Unemployment rate (%)", "Macro", "FRED", "feature", "yes"),
    "GDPC1":    ("Real GDP ($B, chained)", "Macro", "FRED", "feature", "yes"),
    "CPIAUCSL": ("CPI, all urban consumers", "Macro", "FRED", "feature", "yes"),
    "USSTHPI":  ("House price index", "Macro", "FRED", "feature", "yes"),
    "BAA10Y":   ("Baa corporate credit spread (%)", "Macro", "FRED", "feature", "yes"),
    "USREC":    ("NBER recession flag (0/1)", "Macro", "FRED", "feature", "yes"),
    "DRTSCILM": ("Banks tightening C&I loan standards (%)", "Macro", "FRED", "feature", "yes"),
    "NFCI":     ("Chicago Fed financial conditions index", "Macro", "FRED", "feature", "yes"),
    "BOGZ1FL075035503Q": ("Household net worth (flow of funds)", "Macro", "FRED", "feature", "partial"),

    # --- label + label building blocks ---
    "pca_tier":         ("PCA capital tier (well...crit_undercap)", "Label", "derived", "label", ""),
    "is_distressed":    ("Currently undercapitalized-or-worse", "Label", "derived", "label", ""),
    "is_healthy":       ("Currently well/adequate", "Label", "derived", "label", ""),
    "quarters_to_onset":("Quarters until next distress onset", "Label", "derived", "label", ""),
    "onset_4q":         ("TARGET: falls to distress within 4 quarters", "Label", "derived", "label", ""),
}

dict_df = pd.DataFrame(
    [(k, *v) for k, v in FEATURE_DICT.items()],
    columns=["column", "meaning", "category", "source", "role", "lit_backed"],
)

# every panel column should be documented; flag any drift
documented = set(dict_df["column"])
actual = set(panel.columns)
assert not (actual - documented), f"undocumented columns: {actual - documented}"
assert not (documented - actual), f"dict lists missing columns: {documented - actual}"
print(f"all {len(actual)} panel columns documented")

all 64 panel columns documented


### Attach the numbers

Missing %, and, for numeric columns, min / median / max, so the dictionary doubles as a data-quality snapshot. Ranges are the first place cleaning problems show up (e.g. a ratio that should sit in 0-100 running to 20,000).

In [5]:
stats = []
for col in dict_df["column"]:
    s = panel[col]
    row = {"column": col, "missing_pct": round(s.isna().mean() * 100, 2)}
    if pd.api.types.is_numeric_dtype(s) and s.dtype != bool:
        row.update(
            min=round(s.min(), 2),
            median=round(s.median(), 2),
            max=round(s.max(), 2),
        )
    stats.append(row)

stats_df = pd.DataFrame(stats)
dictionary = dict_df.merge(stats_df, on="column")
dictionary

,column,meaning,category,source,role,lit_backed,missing_pct,min,median,max
0,CERT,FDIC certificate number (bank id),ID,financials,id,,0.00,8.00,17027.00,9.139300e+04
1,REPDTE,Reporting quarter-end date,ID,financials,id,,0.00,NaN,NaN,NaN
2,NAME,Bank name,ID,financials,id,,0.00,NaN,NaN,NaN
3,ASSET,Total assets ($000),Size,financials,feature,yes,0.00,1.00,107044.50,4.016571e+09
4,DEP,Total deposits ($000),Size,financials,feature,yes,0.00,0.00,89586.00,2.787994e+09
5,RBC1AAJ,Tier 1 leverage ratio (% avg assets),Capital,financials,feature,yes,0.00,-1524.07,9.50,2.611154e+04
6,RBC1RWAJ,Tier 1 risk-based capital ratio (%),Capital,financials,feature,yes,3.74,-16629.63,14.44,2.746364e+05
7,RBCRWAJ,Total risk-based capital ratio (%),Capital,financials,feature,yes,3.74,-16629.63,15.56,2.746364e+05
8,RBCT1CER,Common equity tier 1 ratio (%),Capital,financials,feature,yes,82.65,-72.90,14.18,7.428333e+04
9,RBCT1J,Tier 1 (core) capital ($000),Capital,financials,feature,yes,0.27,-7789337.00,10532.00,2.957580e+08


### Read-outs

Two quick views off the dictionary: where the biggest gaps are, and the feature count by category.

In [6]:
print("Highest missing % (top 12):")
display(dictionary.sort_values("missing_pct", ascending=False)
        [["column", "meaning", "category", "missing_pct"]].head(12))

print("\nFeature columns by category:")
display(dictionary[dictionary["role"] == "feature"]
        .groupby("category").size().rename("n_features").sort_values(ascending=False))

Highest missing % (top 12):


,column,meaning,category,missing_pct
62,quarters_to_onset,Quarters until next distress onset,Label,99.32
8,RBCT1CER,Common equity tier 1 ratio (%),Capital,82.65
35,SCAA,Available-for-sale securities ($000),Funding,24.50
34,SCHF,"HTM securities, fair value ($000)",Funding,24.50
36,SCAF,"AFS securities, fair value ($000)",Funding,18.61
33,SCHA,Held-to-maturity securities ($000),Funding,18.61
27,DEPUNA,Uninsured deposits ($000),Funding,17.66
6,RBC1RWAJ,Tier 1 risk-based capital ratio (%),Capital,3.74
7,RBCRWAJ,Total risk-based capital ratio (%),Capital,3.74
63,onset_4q,TARGET: falls to distress within 4 quarters,Label,1.88



Feature columns by category:


category
Macro            12
Funding          10
Asset quality     8
Capital           8
Loan mix          6
Earnings          5
Static            4
Size              2
Name: n_features, dtype: int64

## Concern: missing values

The read-out flags a handful of columns with real gaps. Before deciding any fix, diagnose *why* each is blank, a blank can be a structural era gap, a meaningful signal, or a genuine error, and they need different treatment.

The tell for a **structural** gap: missingness is near-total in early years and drops to near-zero after a specific date, the field simply didn't exist on the Call Report yet. A **random** gap would be spread evenly across years.

In [7]:
# columns with non-trivial missingness, worst first
gappy = (dictionary[dictionary["missing_pct"] > 1]
         .sort_values("missing_pct", ascending=False)["column"].tolist())

panel["_yr"] = panel["REPDTE"].dt.year

def first_year_populated(col, thresh=50):
    """First year the column is <thresh% missing, i.e. when it starts being reported."""
    by_yr = panel.groupby("_yr")[col].apply(lambda s: s.isna().mean() * 100)
    ok = by_yr[by_yr < thresh]
    return int(ok.index.min()) if len(ok) else None

diag = pd.DataFrame({
    "column": gappy,
    "meaning": [FEATURE_DICT[c][0] for c in gappy],
    "missing_pct": [round(panel[c].isna().mean() * 100, 1) for c in gappy],
    "starts_reporting": [first_year_populated(c) for c in gappy],
})
diag

,column,meaning,missing_pct,starts_reporting
0,quarters_to_onset,Quarters until next distress onset,99.3,NaN
1,RBCT1CER,Common equity tier 1 ratio (%),82.7,2014.0
2,SCHF,"HTM securities, fair value ($000)",24.5,1994.0
3,SCAA,Available-for-sale securities ($000),24.5,1994.0
4,SCHA,Held-to-maturity securities ($000),18.6,1994.0
5,SCAF,"AFS securities, fair value ($000)",18.6,1994.0
6,DEPUNA,Uninsured deposits ($000),17.7,1993.0
7,RBC1RWAJ,Tier 1 risk-based capital ratio (%),3.7,1990.0
8,RBCRWAJ,Total risk-based capital ratio (%),3.7,1990.0
9,onset_4q,TARGET: falls to distress within 4 quarters,1.9,1990.0


In [8]:
# Confirm the structural read: for each gappy column, missing% in its pre-start era
# vs. after. Structural = near-100% before, near-0% after.
for col in gappy:
    start = first_year_populated(col)
    if start is None:
        print(f"{col:20s} never crosses 50% populated")
        continue
    before = panel.loc[panel["_yr"] < start, col].isna().mean() * 100
    after = panel.loc[panel["_yr"] >= start, col].isna().mean() * 100
    print(f"{col:20s} before {start}: {before:5.1f}% missing | {start}+: {after:4.1f}% missing")

quarters_to_onset    never crosses 50% populated
RBCT1CER             before 2014: 100.0% missing | 2014+: 16.9% missing
SCHF                 before 1994: 100.0% missing | 1994+:  7.2% missing
SCAA                 before 1994: 100.0% missing | 1994+:  7.2% missing
SCHA                 before 1994: 100.0% missing | 1994+:  0.0% missing
SCAF                 before 1994: 100.0% missing | 1994+:  0.0% missing
DEPUNA               before 1993: 100.0% missing | 1993+:  3.9% missing
RBC1RWAJ             before 1990:   nan% missing | 1990+:  3.7% missing
RBCRWAJ              before 1990:   nan% missing | 1990+:  3.7% missing


onset_4q             before 1990:   nan% missing | 1990+:  1.9% missing
DRTSCILM             before 1990:   nan% missing | 1990+:  1.3% missing


### Verdict on missing values

Every gappy column follows the structural pattern, near-total missing before a start year, near-zero after, so these are **era gaps, not errors**. The field was added to the Call Report at a known date:

| Column(s) | Starts | Reason |
|---|---|---|
| `RBCT1CER` (CET1 ratio) | ~2014 | Basel III introduced the common-equity-tier-1 measure |
| `SCHA / SCHF / SCAA / SCAF` (securities) | ~1994 | HTM/AFS split (FAS 115) took effect 1994 |
| `DEPUNA` (uninsured deposits) | ~1993 | Reporting threshold change |
| `RBC1RWAJ / RBCRWAJ` (capital) | 1990 | Small residual, already handled (CBLR, concern #4) |

`quarters_to_onset` shows as mostly missing too, but that is by design, it is only defined for banks that later hit distress, so it is a label building block, not a feature gap.

**Decision: keep the blanks as `NaN`, do not impute here.**

- The gaps carry real information (which reporting era a row is from); filling them with a made-up number would erase that.
- The *handling* choice is model-specific and belongs in the modeling step: tree models (random forest, gradient boosting) take `NaN` natively; logistic regression will need either an era-aware imputation or these columns dropped for the pre-start years.
- Nothing to fix in the saved table. This is logged as understood, not open.

## Concern #3: merged-away banks vs. healthy survivors

When a bank's charter ends by **merger** (not failure), it leaves the panel. For its last few quarters we can't observe the full 4-quarter forward window, so those healthy rows get labeled `onset_4q = 0`, a confident \"stayed safe\", when the honest answer is *unknown* (censored).

Most merges are healthy banks selling for business reasons (genuine negatives), but some are weak banks absorbed to dodge failure (mislabeled as clean). We can't easily tell which, so we **flag** the affected rows rather than trust or drop them.

Steps:
1. Classify each bank's terminal fate: **active / failed / exited** (merged or absorbed), using `institutions` (ACTIVE, ENDEFYMD) and `failures`.
2. Flag healthy `onset_4q = 0` rows within 4 quarters of a non-failure exit as `near_merge_exit`.
3. Keep every row. Modeling decides whether to drop, keep, or use the flag.

In [9]:
# --- 1. terminal fate per bank ---
inst = institutions.copy()
inst["ENDEFYMD_p"] = pd.to_datetime(inst["ENDEFYMD"], format="%m/%d/%Y", errors="coerce")
fate = inst[["CERT", "ACTIVE", "ENDEFYMD_p"]].drop_duplicates("CERT")

failed_certs = set(failures["CERT"].unique())

def classify(r):
    if r["ACTIVE"] == 1:
        return "active"
    if r["CERT"] in failed_certs:
        return "failed"
    return "exited"  # merged / absorbed / voluntary close

fate["fate"] = fate.apply(classify, axis=1)

# restrict to banks that actually appear in the panel
fate_panel = fate[fate["CERT"].isin(panel["CERT"].unique())]
print("Terminal fate of the", f"{fate_panel['CERT'].nunique():,}", "banks in the panel:")
print(fate_panel["fate"].value_counts())

Terminal fate of the 19,260 banks in the panel:
fate
exited    13423
active     4255
failed     1582
Name: count, dtype: int64


In [10]:
# --- 2. flag healthy negatives within 4 quarters of a non-failure exit ---
panel = panel.merge(fate_panel[["CERT", "fate", "ENDEFYMD_p"]], on="CERT", how="left")

# quarters from this filing to the bank's charter end (approx, 91.25 days/quarter)
q_to_end = (panel["ENDEFYMD_p"] - panel["REPDTE"]).dt.days / 91.25

panel["near_merge_exit"] = (
    (panel["is_healthy"] == True)      # currently healthy
    & (panel["onset_4q"] == False)     # labeled "stayed safe"
    & (panel["fate"] == "exited")      # bank later merged/absorbed, did not fail
    & (q_to_end >= 0) & (q_to_end <= 4)  # within the unobservable forward window
)

n_flag = int(panel["near_merge_exit"].sum())
n_neg = int(((panel["is_healthy"] == True) & (panel["onset_4q"] == False)).sum())
print(f"flagged near_merge_exit: {n_flag:,} rows "
      f"({n_flag / n_neg * 100:.2f}% of healthy negatives)")

# drop the scratch fate columns; keep only the flag
panel = panel.drop(columns=["fate", "ENDEFYMD_p"])

flagged near_merge_exit: 48,694 rows (3.97% of healthy negatives)


### Verdict on concern #3

Of the ~19,500 banks, most (69%) eventually merged, normal industry consolidation, not distress. Failures are a separate, smaller group. The label problem is confined to the **~4% of healthy negatives** flagged as `near_merge_exit`: their \"stayed safe\" label is really *unobserved*, because the bank left before the 4-quarter window closed.

**Fix applied: flag, don't drop.** Added a boolean `near_merge_exit` column; every row is kept.

- Modeling can drop them (`panel[~panel.near_merge_exit]`), keep them, or use the flag as a feature, and report the sensitivity both ways.
- Failures are deliberately *not* flagged: a bank failing is a real distress outcome, not censoring.
- This turns a silent label bias into an explicit, testable choice.

## Concern #6: ROA distorted by mergers and tiny denominators

ROA (return on assets) is a core earnings feature, normally around 1%. Its tails are heavy (min −762%, max +562%), so before trusting it, check what drives the extremes.

**The two tails mean opposite things:**

| Tail | What it is | Evidence | Action |
|---|---|---|---|
| Extreme **positive** (> ~15%) | Artifact, a one-time M&A bargain-purchase gain, or a tiny/de-novo bank with a near-zero denominator. Not organic earnings. | 99% currently healthy; 52% have assets < \\$25M; First-Citizens' ROA jumped 1.2% → 23.6% the quarter it absorbed SVB | **Flag** as artifact |
| Extreme **negative** | Real distress, large losses | 33% currently distressed at ROA < −20% | **Leave alone**, genuine signal |

The acquisition-event join (matching merger dates from `history`) caught only ~0.4% of the positive spikes, too unreliable to gate on, so the flag keys on **implausible magnitude**: a quarterly ROA above +15% is not organically achievable regardless of the exact cause.

**Fix: flag only, no numbers changed.** Add a boolean `roa_artifact` marking the implausible positive spikes. Nothing is capped or deleted; raw `ROA` is untouched. Modeling can drop those rows, or cap them later if a linear model needs it. Negatives are deliberately not flagged, they are real losses, not artifacts.

In [11]:
# flag only -- no numbers changed. Mark the implausible positive spikes as likely
# artifacts (M&A gains / tiny denominators). Negatives are left alone: they are real losses.
ROA_ARTIFACT_HI = 15.0

panel["roa_artifact"] = panel["ROA"] > ROA_ARTIFACT_HI

n_flag = int(panel["roa_artifact"].sum())
print(f"flagged roa_artifact (ROA > {ROA_ARTIFACT_HI}%): {n_flag:,} rows "
      f"({panel['roa_artifact'].mean()*100:.2f}%)")
print(f"  raw ROA left unchanged, range still [{panel['ROA'].min():.1f}, {panel['ROA'].max():.1f}]")
print(f"  of flagged rows, {panel.loc[panel['roa_artifact'], 'is_healthy'].mean()*100:.0f}% are currently healthy (confirms artifact, not distress)")

# sanity: First-Citizens SVB quarter is flagged, value untouched
fc = panel[(panel["CERT"] == 11063) & (panel["REPDTE"] == "2023-03-31")]
print(f"\n  check First-Citizens 2023Q1: ROA {fc['ROA'].iloc[0]:.1f}, roa_artifact={bool(fc['roa_artifact'].iloc[0])}")

flagged roa_artifact (ROA > 15.0%): 4,381 rows (0.35%)
  raw ROA left unchanged, range still [-761.9, 562.3]
  of flagged rows, 99% are currently healthy (confirms artifact, not distress)

  check First-Citizens 2023Q1: ROA 23.6, roa_artifact=True


## Concern #2: banks that looked healthy right before failing

A small tail of failed banks show a `well`/`adequate` capital tier on their last filing before failure. Is that a date-matching bug, or real fast failures the capital ratio can't see?

**Finding: real, not a bug.** Of 1,360 failed banks with a pre-failure filing, 88 (6.5%) looked healthy at the last one. The gap from that filing to failure is a normal ~2-month reporting lag (median 62 days, max 125), not a mismatch. Their capital looked genuinely fine (median 11.6%). The 2023 names are the marquee cases: **SVB (16.0% capital), Signature (12.3%), First Republic (12.7%)**, killed by deposit runs, not capital erosion.

**The label is correct.** These banks really were well-capitalized on paper. The gap is a *feature* gap (concern #5), not a label error: capital and asset-quality ratios can't see a liquidity run.

**But the warning was in the data**, just in the funding columns, not the capital ones. SVB into failure:

| Signal | Trajectory | |
|---|---|---|
| Capital ratio | 11.5% → 16.0% | rose, useless |
| Uninsured deposit % | ~86% throughout | standing red flag (normal ~40-50%) |
| Deposit growth | +21% ... then −5%, −6%, −2% | the run building |
| Cash buffer (CHBALR) | 15% → 6% | liquidity draining |

The signal was the deposit-flow **trend** flipping negative, a snapshot misses it, a trend feature catches it.

**Fix: flag only.** Add `fast_failure` marking these healthy-at-last-filing failures, so model evaluation can report performance on them separately (the hardest cases). No label changed.

In [12]:
# identify failed banks whose LAST filing before failure still looked healthy
failed = (failures[["CERT", "FAILDATE"]].dropna()
          .drop_duplicates("CERT"))
failed = failed[failed["CERT"].isin(panel["CERT"].unique())]

pf = panel[["CERT", "REPDTE", "is_healthy"]].merge(failed, on="CERT", how="inner")
pf = pf[pf["REPDTE"] < pf["FAILDATE"]]                       # filings before failure
last_pre = pf.sort_values("REPDTE").groupby("CERT").tail(1)  # the last one

# certs that looked healthy at that last filing = fast failures
fast_certs = set(last_pre.loc[last_pre["is_healthy"] == True, "CERT"])
print(f"failed banks with a pre-failure filing: {last_pre['CERT'].nunique():,}")
print(f"looked healthy at the last one (fast_failure): {len(fast_certs):,} "
      f"({len(fast_certs)/last_pre['CERT'].nunique()*100:.1f}%)")

# flag every row of those banks (whole trajectory belongs to a fast-failure case)
panel["fast_failure"] = panel["CERT"].isin(fast_certs)
print(f"rows flagged fast_failure: {int(panel['fast_failure'].sum()):,}")

# confirm the 2023 marquee names are captured
for cert, name in [(24735, "SVB"), (57053, "Signature"), (59017, "First Republic")]:
    print(f"  {name} (CERT {cert}): fast_failure={cert in fast_certs}")

failed banks with a pre-failure filing: 1,360
looked healthy at the last one (fast_failure): 91 (6.7%)
rows flagged fast_failure: 5,370
  SVB (CERT 24735): fast_failure=True
  Signature (CERT 57053): fast_failure=True
  First Republic (CERT 59017): fast_failure=True


## Concern #8: the last quarters have no outcome to check against

`onset_4q` asks: does this bank fall to undercapitalized within the **next 4 quarters**? Answering it requires those 4 quarters to exist in the data.

The panel ends **2026-03**. So:

| Filing quarter | Window needed | Have it? |
|---|---|---|
| 2025-03 and earlier | through 2026-03 | yes, fully checkable |
| 2025-06 onward | runs past 2026-03 | no, outcome not observed yet |

For the recent rows the label currently reads `0` ("stayed safe") when the honest answer is **unknown**. That is not a small edge case: it is roughly 17,500 rows, and because they are all negatives they quietly inflate any test score computed on that period.

**Fix: set `onset_4q` to blank (NaN) for the unobservable rows.** The panel already uses blank for "cannot be labeled" (rows that are already distressed carry no target), so this puts them in the bucket that already exists rather than inventing a second one. No flag column needed.

Two details:

- A `1` in that window **stays a `1`**. If the bank already fell to undercapitalized inside the part of the window we can see, the answer is confirmed regardless of the missing tail.
- No rows are deleted and no feature values change. The rows keep every predictor, so the model can still score them, they just cannot be graded. That makes them the natural "who looks risky right now" set for the final presentation.

In [13]:
# The 4-quarter forward window needs 4 more quarters of filings to exist.
# Anything filed after (last date - 4 quarters) cannot have its window observed.
LAST_DATE = panel["REPDTE"].max()
LAST_FULL_LABEL_DATE = LAST_DATE - pd.DateOffset(months=12)

print(f"panel ends           {LAST_DATE:%Y-%m}")
print(f"last checkable filing {LAST_FULL_LABEL_DATE:%Y-%m}\n")

# before: label counts per quarter in the affected tail
tail = panel[panel["REPDTE"] > LAST_FULL_LABEL_DATE]
before = (tail.groupby("REPDTE")["onset_4q"]
          .agg(labeled="count", positives="sum").astype(int))

# a 0 in that window means "not seen yet", not "did not happen" -> blank it.
# a 1 is confirmed inside the visible part of the window -> keep it.
unobservable = (panel["REPDTE"] > LAST_FULL_LABEL_DATE) & (panel["onset_4q"] == 0)
n_blanked = int(unobservable.sum())
panel.loc[unobservable, "onset_4q"] = np.nan

after = (panel[panel["REPDTE"] > LAST_FULL_LABEL_DATE]
         .groupby("REPDTE")["onset_4q"]
         .agg(labeled="count", positives="sum").astype(int))

print(f"blanked {n_blanked:,} unobservable negatives "
      f"({n_blanked / len(panel) * 100:.2f}% of the panel)\n")
print(before.join(after, lsuffix="_before", rsuffix="_after"))

panel ends           2026-03
last checkable filing 2025-03



blanked 17,521 unobservable negatives (1.39% of the panel)

            labeled_before  positives_before  labeled_after  positives_after
REPDTE                                                                      
2025-06-30            4452                 5              5                5
2025-09-30            4414                 4              4                4
2025-12-31            4360                 2              2                2
2026-03-31            4306                 0              0                0


## Save the cleaned panel

Drop the scratch `_yr` column and save as `panel_clean.parquet`, the table feature engineering and modeling will read. No rows were deleted and no feature value was altered anywhere in this notebook.

Three boolean flags added:

| Flag | Concern | Meaning |
|---|---|---|
| `near_merge_exit` | #3 | Healthy negative within 4 quarters of a non-failure exit (censored label) |
| `roa_artifact` | #6 | ROA > 15%, implausible positive spike (M&A gain / tiny denominator) |
| `fast_failure` | #2 | Belongs to a bank that looked healthy at its last filing before failing (evaluation tag, not a feature) |

One label correction:

| Column | Concern | Change |
|---|---|---|
| `onset_4q` | #8 | Unobservable negatives after 2025-03 set to blank (unknown), since their 4-quarter window runs past the end of the data. Confirmed positives kept. |

In [14]:
new_flags = ["near_merge_exit", "roa_artifact", "fast_failure"]

# drop scratch column, confirm nothing else changed
out = panel.drop(columns=["_yr"], errors="ignore")
assert out.shape[0] == panel.shape[0], "row count changed, should not happen"

# no "stayed safe" label may survive past the observable window
bad = int(((out["REPDTE"] > LAST_FULL_LABEL_DATE) & (out["onset_4q"] == 0)).sum())
assert bad == 0, f"{bad} unobservable negatives still labeled"

out_path = PROCESSED / "panel_clean.parquet"
out.to_parquet(out_path)

print(f"saved {out.shape[0]:,} rows x {out.shape[1]} cols -> {out_path.name}")
print("\nflags added this notebook:")
print(out[new_flags].sum().rename("n_rows_true").to_frame())
print(f"\nonset_4q labeled rows: {int(out['onset_4q'].notna().sum()):,} "
      f"({int(out['onset_4q'].sum()):,} positives), "
      f"blank: {int(out['onset_4q'].isna().sum()):,}")

saved 1,258,888 rows x 67 cols -> panel_clean.parquet

flags added this notebook:
                 n_rows_true
near_merge_exit        48694
roa_artifact            4381
fast_failure            5370

onset_4q labeled rows: 1,217,743 (8,617 positives), blank: 41,145
